# Decision Intelligence Assistant

This notebook covers data preparation, weak labeling, feature engineering, model comparison, and artifact export for the project.

## 1. Goals

- Build a representative 10k-row sample from the Twitter support dataset.
- Create a transparent weak-labeling rule for ticket priority.
- Compare TF-IDF, engineered, and combined features.
- Select a deployable ML baseline and export its artifacts.
- Collect examples for the final written analysis.

## 2. Imports

In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import re
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
TARGET_SAMPLE_SIZE = 10_000

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_raw_dir = project_root / "data" / "raw"
data_sample_dir = project_root / "data" / "sample"
artifacts_dir = project_root / "artifacts"
backend_path = project_root / "backend"
if str(backend_path) not in sys.path:
    sys.path.append(str(backend_path))

data_sample_dir.mkdir(parents=True, exist_ok=True)
artifacts_dir.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

print(f"Project root: {project_root}")
print(f"Raw data dir: {data_raw_dir}")
print(f"Sample output dir: {data_sample_dir}")


Project root: E:\Extra Courses\AIE Bootcamp\Week 3\Project 3\decision-intelligence-assistant
Raw data dir: E:\Extra Courses\AIE Bootcamp\Week 3\Project 3\decision-intelligence-assistant\data\raw
Sample output dir: E:\Extra Courses\AIE Bootcamp\Week 3\Project 3\decision-intelligence-assistant\data\sample


In [1]:
from pathlib import Path
import shutil

try:
    import kagglehub
except ImportError:
    kagglehub = None

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

raw_dir = project_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

if kagglehub is None:
    print("kagglehub is not installed. Place twcs.csv in data/raw manually.")
else:
    download_path = Path(
        kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")
    )
    print("Downloaded to:", download_path)

    csv_files = list(download_path.rglob("*.csv"))
    print("CSV files found:", [p.name for p in csv_files])

    preferred = next((p for p in csv_files if p.name == "twcs.csv"), None)
    source_csv = preferred or next((p for p in csv_files if p.name == "sample.csv"), None)
    if source_csv is None:
        raise FileNotFoundError("No CSV file was found in the downloaded dataset.")

    target_csv = raw_dir / source_csv.name
    shutil.copy2(source_csv, target_csv)
    print("Copied dataset to:", target_csv)
    if source_csv.name != "twcs.csv":
        print("Warning: using sample.csv fallback, not the full twcs.csv dataset.")


100%|██████████| 169M/169M [00:14<00:00, 12.0MB/s] 

Extracting files...


Downloaded to: C:\Users\Pc\.cache\kagglehub\datasets\thoughtvector\customer-support-on-twitter\versions\10
CSV files found: ['sample.csv', 'twcs.csv']
Copied dataset to: E:\Extra Courses\AIE Bootcamp\Week 3\Project 3\decision-intelligence-assistant\data\raw\sample.csv


## 3. Load Raw Dataset

In [ ]:
def parse_linked_ids(raw_value: Any) -> list[str]:
    """Parse linked tweet identifiers from a raw cell value."""
    if pd.isna(raw_value):
        return []

    raw_text = str(raw_value).strip()
    if not raw_text:
        return []

    return [token for token in re.split(r"[\s,\[\]]+", raw_text) if token]


def to_bool_flag(raw_value: Any) -> bool:
    """Coerce the inbound column to a real boolean flag."""
    if isinstance(raw_value, bool):
        return raw_value
    if pd.isna(raw_value):
        return False

    normalized = str(raw_value).strip().lower()
    return normalized in {"true", "t", "1", "yes"}


def load_raw_dataset(raw_dir: Path) -> tuple[pd.DataFrame, Path]:
    """Load the first supported dataset found in data/raw."""
    candidates = sorted(raw_dir.glob("*.csv")) + sorted(raw_dir.glob("*.parquet"))

    if not candidates:
        raise FileNotFoundError(
            "No raw dataset was found in data/raw. Place the Kaggle Twitter support "
            "CSV or Parquet file there before running this notebook."
        )

    dataset_path = candidates[0]
    if dataset_path.suffix == ".csv":
        dataframe = pd.read_csv(dataset_path)
    else:
        dataframe = pd.read_parquet(dataset_path)

    return dataframe, dataset_path


raw_df, dataset_path = load_raw_dataset(data_raw_dir)
print(f"Loaded dataset: {dataset_path.name}")
print(f"Raw shape: {raw_df.shape}")

required_columns = {"tweet_id", "author_id", "inbound", "text"}
missing_columns = required_columns - set(raw_df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df = raw_df.copy()
df["tweet_id"] = df["tweet_id"].astype(str)
df["author_id"] = df["author_id"].astype(str)
df["text"] = df["text"].fillna("").astype(str)
df["inbound_flag"] = df["inbound"].apply(to_bool_flag)

tweet_to_author = df.set_index("tweet_id")["author_id"].to_dict()
if "response_tweet_id" in df.columns:
    df["brand_hint"] = df["response_tweet_id"].apply(
        lambda value: next(
            (tweet_to_author[tweet_id] for tweet_id in parse_linked_ids(value)
             if tweet_id in tweet_to_author),
            "unknown",
        )
    )
else:
    df["brand_hint"] = "unknown"

customer_tickets = df.loc[df["inbound_flag"]].copy()
customer_tickets["text"] = customer_tickets["text"].str.strip()
customer_tickets = customer_tickets.loc[customer_tickets["text"].ne("")].copy()
customer_tickets["normalized_text"] = (
    customer_tickets["text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
customer_tickets = customer_tickets.drop_duplicates(subset=["tweet_id"])
customer_tickets = customer_tickets.drop_duplicates(subset=["normalized_text"])

customer_tickets["text_length"] = customer_tickets["text"].str.len()
customer_tickets["word_count"] = customer_tickets["text"].str.split().str.len()

print(f"Customer-ticket shape after cleaning: {customer_tickets.shape}")
display(customer_tickets.head())


## 4. Build 10k Representative Sample

In [ ]:
def build_representative_sample(
    dataframe: pd.DataFrame,
    target_size: int,
    random_state: int = 42,
) -> pd.DataFrame:
    """Create a reproducible ticket sample that preserves brand variety."""
    if dataframe.empty:
        raise ValueError("Cannot sample from an empty dataframe.")

    sample_size = min(target_size, len(dataframe))
    working_df = dataframe.copy()
    working_df["brand_hint"] = working_df["brand_hint"].fillna("unknown")

    group_sizes = working_df["brand_hint"].value_counts()
    exact_allocations = group_sizes / group_sizes.sum() * sample_size
    allocations = np.floor(exact_allocations).astype(int)
    allocations[allocations == 0] = 1

    while allocations.sum() > sample_size:
        largest_group = allocations.idxmax()
        if allocations[largest_group] > 1:
            allocations[largest_group] -= 1
        else:
            break

    while allocations.sum() < sample_size:
        remainders = exact_allocations - allocations
        next_group = remainders.idxmax()
        allocations[next_group] += 1

    samples = []
    for brand_hint, group_df in working_df.groupby("brand_hint", sort=False):
        requested_n = min(allocations.get(brand_hint, 0), len(group_df))
        if requested_n == 0:
            continue
        sampled_group = group_df.sample(n=requested_n, random_state=random_state)
        samples.append(sampled_group)

    sample_df = pd.concat(samples, ignore_index=True)
    if len(sample_df) > sample_size:
        sample_df = sample_df.sample(n=sample_size, random_state=random_state)

    return sample_df.sort_values("tweet_id").reset_index(drop=True)


sample_df = build_representative_sample(
    dataframe=customer_tickets,
    target_size=TARGET_SAMPLE_SIZE,
    random_state=SEED,
)

sample_output_path = data_sample_dir / "customer_support_sample.csv"
sample_df.to_csv(sample_output_path, index=False)

print(f"Saved representative sample to: {sample_output_path}")
print(f"Sample shape: {sample_df.shape}")
print(
    "Brand coverage in sample:",
    sample_df["brand_hint"].nunique(),
)

sample_summary = pd.DataFrame(
    {
        "full_dataset_count": customer_tickets["brand_hint"].value_counts(),
        "sample_count": sample_df["brand_hint"].value_counts(),
    }
).fillna(0).astype(int)
sample_summary["sample_share"] = (
    sample_summary["sample_count"] / sample_summary["sample_count"].sum()
).round(4)

display(sample_summary.head(15))
display(sample_df[["tweet_id", "author_id", "brand_hint", "text"]].head(10))


## 5. Weak Labeling Rule

In [ ]:
from backend.app.services.features import extract_features, weak_label_priority

sample_df["priority_label"] = sample_df["text"].apply(weak_label_priority)
label_distribution = sample_df["priority_label"].value_counts().rename_axis("priority").reset_index(name="count")
label_distribution["share"] = (label_distribution["count"] / len(sample_df)).round(3)

print("Weak-label distribution")
display(label_distribution)

display(
    sample_df[["tweet_id", "brand_hint", "text", "priority_label"]]
    .sort_values("priority_label")
    .head(12)
)


## 6. Feature Engineering

In [ ]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

feature_rows = [extract_features(text).__dict__ for text in sample_df["text"]]
engineered_df = pd.DataFrame(feature_rows)

vectorizer = TfidfVectorizer(
    max_features=20_000,
    ngram_range=(1, 2),
    min_df=1,
    stop_words="english",
)
tfidf_matrix = vectorizer.fit_transform(sample_df["text"])
engineered_matrix = engineered_df.to_numpy(dtype=float)
combined_matrix = hstack([tfidf_matrix, engineered_matrix])

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(sample_df["priority_label"])

print("TF-IDF shape:", tfidf_matrix.shape)
print("Engineered feature shape:", engineered_matrix.shape)
display(engineered_df.describe().T)


## 7. Model Comparison

In [ ]:
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

feature_sets = {
    "tfidf_only": tfidf_matrix,
    "engineered_only": engineered_matrix,
    "tfidf_engineered": combined_matrix,
}
models = {
    "LogisticRegression": LogisticRegression(max_iter=3000, class_weight="balanced"),
    "LinearSVC": LinearSVC(max_iter=20000, class_weight="balanced"),
    "RandomForestClassifier": RandomForestClassifier(
        n_estimators=200,
        random_state=SEED,
        class_weight="balanced",
        n_jobs=1,
    ),
}

results = []
trained_models = {}

for feature_name, features in feature_sets.items():
    x_train, x_test, y_train, y_test = train_test_split(
        features,
        y,
        test_size=0.2,
        random_state=SEED,
        stratify=y,
    )
    for model_name, model in models.items():
        start = time.perf_counter()
        model.fit(x_train, y_train)
        predictions = model.predict(x_test)
        latency_ms = ((time.perf_counter() - start) / max(len(y_test), 1)) * 1000
        macro_f1 = f1_score(y_test, predictions, average="macro")
        results.append(
            {
                "feature_set": feature_name,
                "model": model_name,
                "accuracy": accuracy_score(y_test, predictions),
                "macro_f1": macro_f1,
                "latency_ms_per_prediction": latency_ms,
            }
        )
        trained_models[(feature_name, model_name)] = (model, x_test, y_test, predictions)

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
display(results_df)

best_row = results_df.loc[results_df["feature_set"].eq("tfidf_engineered")].iloc[0]
best_key = (best_row["feature_set"], best_row["model"])
best_model, x_test, y_test, predictions = trained_models[best_key]

print("Selected model:", best_key)
print(classification_report(
    y_test,
    predictions,
    labels=list(range(len(label_encoder.classes_))),
    target_names=label_encoder.classes_,
    zero_division=0,
))

display(pd.DataFrame(
    confusion_matrix(y_test, predictions, labels=list(range(len(label_encoder.classes_)))),
    index=label_encoder.classes_,
    columns=label_encoder.classes_,
))


## 8. Error Analysis

In [ ]:
error_cases = sample_df.iloc[getattr(x_test, "indices", np.arange(min(len(sample_df), len(y_test))))[:0]].copy()
comparison_df = pd.DataFrame(
    {
        "actual": label_encoder.inverse_transform(y_test),
        "predicted": label_encoder.inverse_transform(predictions),
    }
)
comparison_df["is_correct"] = comparison_df["actual"].eq(comparison_df["predicted"])

print("Prediction correctness summary")
display(comparison_df["is_correct"].value_counts().rename_axis("correct").reset_index(name="count"))
print("Important limitation: labels are weak supervision from rules, not human annotations.")


## 9. Export Artifacts

In [ ]:
import json
import joblib

artifacts_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, artifacts_dir / "priority_model.joblib")
joblib.dump(vectorizer, artifacts_dir / "vectorizer.joblib")
joblib.dump(label_encoder, artifacts_dir / "label_encoder.joblib")

metadata = {
    "selected_feature_set": best_key[0],
    "selected_model": best_key[1],
    "rows": int(len(sample_df)),
    "label_distribution": sample_df["priority_label"].value_counts().to_dict(),
    "model_comparison": results_df.to_dict(orient="records"),
    "recommendation": (
        "Use the trained ML model for high-volume routing because latency and cost are much lower; "
        "use the LLM for low-confidence or high-risk cases where reasoning is valuable."
    ),
}
(artifacts_dir / "model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Exported artifacts to:", artifacts_dir)
print(json.dumps(metadata, indent=2)[:1500])
